## Layer 1

In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [2]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a financial research assistant. Be concise."},
        {"role": "user", "content": "Should I invest in NVIDIA right now? Give me 3 bullet points."}
    ]
)

In [3]:
print(response.choices[0].message.content)

1. **Strong Growth Potential**: NVIDIA is a leader in AI and graphics processing units (GPUs), with increasing demand across various sectors like gaming, data centers, and AI research.

2. **Valuation Considerations**: Assess the current valuation metrics like P/E ratio and compare them with industry peers. High valuations may pose risks if growth slows.

3. **Market Volatility**: The tech sector can be volatile; consider your risk tolerance and investment horizon before making a decision.


In [5]:
input_cost  = (response.usage.prompt_tokens / 1_000_000) * 0.150
output_cost = (response.usage.completion_tokens / 1_000_000) * 0.600

print(f"Prompt tokens:     {response.usage.prompt_tokens}")
print(f"Completion tokens: {response.usage.completion_tokens}")
print(f"Cost:              ${input_cost + output_cost:.6f}")

Prompt tokens:     36
Completion tokens: 99
Cost:              $0.000065


In [6]:
response2 = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a financial research assistant."},
        {"role": "user", "content": "What is NVIDIA's exact current stock price and P/E ratio?"}
    ]
)
print(response2.choices[0].message.content)

I'm unable to provide real-time data, including NVIDIA's current stock price and P/E ratio, as my training only includes data up to October 2023. To get the most accurate and up-to-date information, please check a financial news website, stock market application, or a stock exchange platform.


## Layer 2

In [7]:
import os
import json
import yfinance as yf
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [8]:
def get_stock_data(ticker: str) -> dict:
    """
    Fetches real stock data from Yahoo Finance.
    Returns price, P/E ratio, market cap, and 52-week range.
    """
    stock = yf.Ticker(ticker)
    info = stock.info
    
    return {
        "ticker": ticker,
        "current_price": info.get("currentPrice"),
        "pe_ratio": info.get("trailingPE"),
        "market_cap": info.get("marketCap"),
        "52_week_high": info.get("fiftyTwoWeekHigh"),
        "52_week_low": info.get("fiftyTwoWeekLow"),
        "revenue_growth": info.get("revenueGrowth"),
        "profit_margins": info.get("profitMargins"),
    }

In [9]:
nvda_data = get_stock_data("NVDA")
print(json.dumps(nvda_data, indent=2))

{
  "ticker": "NVDA",
  "current_price": 225.32,
  "pe_ratio": 46.077713,
  "market_cap": 5457368842240,
  "52_week_high": 236.54,
  "52_week_low": 129.16,
  "revenue_growth": 0.732,
  "profit_margins": 0.55603004
}


In [10]:
tools = [
    {
        "type": "function",
        "function":{
            "name": "get_stock_data",
            "description": "Fetches real stock data from Yahoo Finance. Returns price, P/E ratio, market cap, and 52-week range.",
            "parameters": {
                "type": "object",
                "properties": {
                    "ticker": {
                        "type": "string",
                        "description": "The stock ticker symbol (e.g., 'NVDA')."
                    }
                },
                "required": ["ticker"]
             }  
        }
    }
]

In [11]:
messages = [
    {"role": "system", "content": "You are a financial research assistant. Use tools to get real data before making any claims."},
    {"role": "user", "content": "What is NVIDIA's current stock price and P/E ratio? Is it overvalued?"}
]

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    tools=tools,
    tool_choice="auto"
)

print("Stop reason: ", response.choices[0].finish_reason)
print("Model response: ", response.choices[0].message.content)

Stop reason:  tool_calls
Model response:  None


In [12]:
tool_call = response.choices[0].message.tool_calls[0]

print("Tool name:", tool_call.function.name)
print("Arguments:", tool_call.function.arguments)
print("Call ID:", tool_call.id)

Tool name: get_stock_data
Arguments: {"ticker":"NVDA"}
Call ID: call_cbwnGvwoUfwST2GA4WsBawLf


In [13]:
import json

# Extract what the model wants to call
tool_call = response.choices[0].message.tool_calls[0]
function_name = tool_call.function.name
arguments = json.loads(tool_call.function.arguments)

# Actually call the function
if function_name == "get_stock_data":
    result = get_stock_data(**arguments)

print("Function result:", result)

Function result: {'ticker': 'NVDA', 'current_price': 225.32, 'pe_ratio': 46.077713, 'market_cap': 5457368842240, '52_week_high': 236.54, '52_week_low': 129.16, 'revenue_growth': 0.732, 'profit_margins': 0.55603004}


In [14]:
messages_with_tool_result = messages + [
    response.choices[0].message,  # The model's message that includes the tool call
    {"role": "tool", "tool_call_id": tool_call.id, "content": json.dumps(result)}  # The tool's response
]

messages_with_tool_result

[{'role': 'system',
  'content': 'You are a financial research assistant. Use tools to get real data before making any claims.'},
 {'role': 'user',
  'content': "What is NVIDIA's current stock price and P/E ratio? Is it overvalued?"},
 ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_cbwnGvwoUfwST2GA4WsBawLf', function=Function(arguments='{"ticker":"NVDA"}', name='get_stock_data'), type='function')]),
 {'role': 'tool',
  'tool_call_id': 'call_cbwnGvwoUfwST2GA4WsBawLf',
  'content': '{"ticker": "NVDA", "current_price": 225.32, "pe_ratio": 46.077713, "market_cap": 5457368842240, "52_week_high": 236.54, "52_week_low": 129.16, "revenue_growth": 0.732, "profit_margins": 0.55603004}'}]

In [15]:
final_response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages_with_tool_result,
    tools=tools,
)

In [17]:
print("\n +++++++++++ Final response from model after tool call +++++++++++\n")
print(final_response.choices[0].message.content)


 +++++++++++ Final response from model after tool call +++++++++++

NVIDIA's current stock price is $225.32, and its P/E (Price-to-Earnings) ratio is approximately 46.08.

### Valuation Perspective:
A P/E ratio of 46.08 can indicate that a stock may be overvalued, especially when compared to the average market P/E, which typically ranges from 15 to 25 for many industries. However, high P/E ratios can also reflect high expectations for growth, particularly in a company like NVIDIA, known for its innovation in fields like gaming and artificial intelligence.

To determine if it's overvalued, one should also consider:

1. **Industry Comparisons:** How does NVIDIA's P/E ratio compare to its peers?
2. **Growth Rates:** NVIDIA has a revenue growth rate of 73.2%, which is significantly high and can justify higher P/E ratios.
3. **Market Sentiment:** Market conditions and investor sentiment can also influence perceptions of value.

Overall, while the P/E ratio suggests potential overvaluation,